In [1]:
 import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/patient_level_final.csv', index_col='Patient_ID')

In [3]:
df.shape

(40336, 79)

In [4]:
X = df.drop(columns=['SepsisLabel'])
y = df['SepsisLabel']

print(X.shape, y.shape)

(40336, 78) (40336,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(32268, 78) (8068, 78)
SepsisLabel
0    29922
1     2346
Name: count, dtype: int64
SepsisLabel
0    7482
1     586
Name: count, dtype: int64


In [6]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

print("Training done")

Training done


c:\Users\Tanishq Raj Singh\anaconda3\envs\booksenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

print("Training done")

Training done


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = log_reg.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[5810 1672]
 [ 147  439]]
              precision    recall  f1-score   support

           0       0.98      0.78      0.86      7482
           1       0.21      0.75      0.33       586

    accuracy                           0.77      8068
   macro avg       0.59      0.76      0.60      8068
weighted avg       0.92      0.77      0.83      8068



In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[7436   46]
 [ 326  260]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      7482
           1       0.85      0.44      0.58       586

    accuracy                           0.95      8068
   macro avg       0.90      0.72      0.78      8068
weighted avg       0.95      0.95      0.95      8068



In [10]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(scale_pos_weight=scale, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

[[7292  190]
 [ 187  399]]
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      7482
           1       0.68      0.68      0.68       586

    accuracy                           0.95      8068
   macro avg       0.83      0.83      0.83      8068
weighted avg       0.95      0.95      0.95      8068



In [11]:
print("Model comparison summary (sepsis class only):")
print(f"Logistic Regression - Precision: 0.21, Recall: 0.75, F1: 0.33")
print(f"Random Forest        - Precision: 0.85, Recall: 0.44, F1: 0.58")
print(f"XGBoost               - Precision: 0.68, Recall: 0.68, F1: 0.68")
print()
print("Selected model: XGBoost (best F1-score, most balanced precision/recall)")

best_model = xgb

Model comparison summary (sepsis class only):
Logistic Regression - Precision: 0.21, Recall: 0.75, F1: 0.33
Random Forest        - Precision: 0.85, Recall: 0.44, F1: 0.58
XGBoost               - Precision: 0.68, Recall: 0.68, F1: 0.68

Selected model: XGBoost (best F1-score, most balanced precision/recall)
